# Submission — Logistic Regression Model

- Men: LR with Elo + Seed + POM + Off_Eff + Win_pct + Elo*Seed interaction (C=0.2)
- Women: LR with Elo + Seed + Win_pct (C=1.0)
- Train on <2022, holdout eval on 2022-2025
- Falls back to Elo for teams missing features

## Experiment results (LOSO / Holdout / CoefStab)
- Men baseline (3 feat, C=1.0): 0.18834 / 0.19504 / 96.8%
- **Men best (5 feat + Elo*Seed, C=0.2): 0.18788 / 0.19199 / 94.4%**
- Women baseline (2 feat, C=1.0): 0.14639 / 0.14030 / 100%
- **Women best (3 feat, C=1.0): 0.14466 / 0.14598 / 100%**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_men_data, load_women_data
from src.feature_engineering import *
from src.model import brier_score
from src.model_logreg import (
    train_logreg, predict_logreg, print_logreg_coefficients, add_interactions,
    BEST_MEN_FEATURES, BEST_MEN_INTERACTIONS, BEST_MEN_C,
    BEST_WOMEN_FEATURES, BEST_WOMEN_C,
)
from src.submission import generate_submission_logreg

## Load Data & Build Features

In [2]:
data_dir = f'{project_root}/data'

# Men
m_data = load_men_data(data_dir)
m_elo = compute_elo_ratings(m_data['MComSsn'])
m_seeds = parse_seeds(m_data['MTrnySeeds'])
m_stats = compute_season_stats(m_data['MDetSsn'])
massey = compute_massey_features(m_data['MOrdinals'])
m_conf = compute_conference_strength(m_data['MConf'], m_elo)
m_eff = compute_efficiency(m_data['MDetSsn'])
m_features = build_team_features(m_elo, m_seeds, m_stats, massey, m_conf, efficiency_df=m_eff)

# Women
w_data = load_women_data(data_dir)
w_elo = compute_elo_ratings(w_data['WComSsn'])
w_seeds = parse_seeds(w_data['WTrnySeeds'])
w_stats = compute_season_stats(w_data['WDetSsn'])
w_conf = compute_conference_strength(w_data['WConf'], w_elo)
w_eff = compute_efficiency(w_data['WDetSsn'])
w_features = build_team_features(w_elo, w_seeds, stats_df=w_stats, conf_df=w_conf, efficiency_df=w_eff)

print('Features loaded.')

Features loaded.


## Build Matchups & Holdout Eval

In [3]:
# Feature sets (from experiment results)
MEN_FEAT = BEST_MEN_FEATURES      # ['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff', 'Off_Eff_diff', 'Win_pct_diff']
MEN_INT = BEST_MEN_INTERACTIONS   # [('Elo_diff', 'SeedNum_diff')]
MEN_C = BEST_MEN_C                # 0.2
WOMEN_FEAT = BEST_WOMEN_FEATURES  # ['Elo_diff', 'SeedNum_diff', 'Win_pct_diff']
WOMEN_C = BEST_WOMEN_C            # 1.0

# Build matchup matrices
m_matchups = build_matchup_matrix(m_data['MDetTrny'], m_features)
w_matchups = build_matchup_matrix(w_data['WDetTrny'], w_features)

# Add interaction features for men
m_matchups, men_int_cols = add_interactions(m_matchups, MEN_INT)
MEN_FEAT_ALL = MEN_FEAT + men_int_cols
print(f'Men features: {MEN_FEAT_ALL}')
print(f'Women features: {WOMEN_FEAT}')

m_train = m_matchups[m_matchups['Season'] < 2022]
m_test = m_matchups[m_matchups['Season'] >= 2022]
w_train = w_matchups[w_matchups['Season'] < 2022]
w_test = w_matchups[w_matchups['Season'] >= 2022]

print(f'Men: {len(m_train)} train, {len(m_test)} test')
print(f'Women: {len(w_train)} train, {len(w_test)} test')

Men features: ['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff', 'Off_Eff_diff', 'Win_pct_diff', 'EloxSeedNum_diff']
Women features: ['Elo_diff', 'SeedNum_diff', 'Win_pct_diff']
Men: 1181 train, 268 test
Women: 693 train, 268 test


In [4]:
# Train on <2022, evaluate on 2022-2025
m_model, m_scaler, _ = train_logreg(m_train, MEN_FEAT_ALL, C=MEN_C)
m_preds = predict_logreg(m_model, m_scaler, m_test[MEN_FEAT_ALL].values)

w_model, w_scaler, _ = train_logreg(w_train, WOMEN_FEAT, C=WOMEN_C)
w_preds = predict_logreg(w_model, w_scaler, w_test[WOMEN_FEAT].values)

# Elo baseline for comparison
def get_elo_probs(test_df, elo_df):
    elo_a = test_df.merge(elo_df, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'])['Elo']
    elo_b = test_df.merge(elo_df, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'])['Elo']
    return elo_to_prob(elo_a.values, elo_b.values)

m_elo_probs = get_elo_probs(m_test, m_elo)
w_elo_probs = get_elo_probs(w_test, w_elo)

print(f'Men holdout Brier (LR):       {brier_score(m_test["Target"].values, m_preds):.5f}')
print(f'Men holdout Brier (pure Elo):  {brier_score(m_test["Target"].values, m_elo_probs):.5f}')
print()
print(f'Women holdout Brier (LR):       {brier_score(w_test["Target"].values, w_preds):.5f}')
print(f'Women holdout Brier (pure Elo):  {brier_score(w_test["Target"].values, w_elo_probs):.5f}')

Men holdout Brier (LR):       0.19199
Men holdout Brier (pure Elo):  0.20259

Women holdout Brier (LR):       0.14598
Women holdout Brier (pure Elo):  0.15853


In [5]:
print('Men coefficients:')
print_logreg_coefficients(m_model, MEN_FEAT_ALL)
print()
print('Women coefficients:')
print_logreg_coefficients(w_model, WOMEN_FEAT)

Men coefficients:
Logistic Regression Coefficients:
  Rank_POM_diff                  -0.6127
  Elo_diff                       +0.6104
  SeedNum_diff                   -0.3035
  Off_Eff_diff                   +0.1217
  EloxSeedNum_diff               +0.1049
  Win_pct_diff                   -0.0728

Women coefficients:
Logistic Regression Coefficients:
  Elo_diff                       +2.0697
  SeedNum_diff                   -0.8546
  Win_pct_diff                   -0.7844


## Generate Submission

In [6]:
# Elo lookup for fallback
all_elos = pd.concat([m_elo, w_elo], ignore_index=True)
elo_lookup = all_elos.set_index(['Season', 'TeamID'])['Elo'].to_dict()

In [7]:
sub = generate_submission_logreg(
    f'{project_root}/data/SampleSubmissionStage1.csv',
    m_model, m_scaler, m_features, MEN_FEAT_ALL,
    w_model, w_scaler, w_features, WOMEN_FEAT,
    elo_lookup,
    men_interaction_pairs=MEN_INT,
)
sub.to_csv(f'{project_root}/submissions/stage1_logreg.csv', index=False)
print('Saved to submissions/stage1_logreg.csv')

Total matchups: 519144 (men: 261013, women: 258131)
LR predictions: 18224, Elo fallback: 500920
Pred range: [0.0200, 0.9800]
Pred mean: 0.4936
Saved to submissions/stage1_logreg.csv
